In [ ]:
%pip install geopandas shapely pyproj rasterio contextily folium scikit-learn jupyterlab xgboost


[notice] A new release of pip is available: 23.1.2 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [53]:
%pip list

Package                   Version
------------------------- --------------
affine                    2.4.0
anyio                     4.9.0
appnope                   0.1.4
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 3.0.0
async-lru                 2.0.5
attrs                     25.3.0
babel                     2.17.0
beautifulsoup4            4.13.4
bleach                    6.2.0
branca                    0.8.1
certifi                   2025.4.26
cffi                      1.17.1
charset-normalizer        3.4.2
click                     8.2.0
click-plugins             1.1.1
cligj                     0.7.2
comm                      0.2.2
contextily                1.6.2
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.14
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.0
fastjsonschema            2.21.1
folium          

In [ ]:
# import requests

# url    = "https://phl.carto.com/api/v2/sql"
# params = {
#     "q": """
#       SELECT
#         *,
#         ST_AsGeoJSON(the_geom)::json AS geometry
#       FROM assessments
#       WHERE year > 2022
#     """,
#     "format": "GeoJSON"
# }

# resp = requests.get(url, params=params)
# resp.raise_for_status()

# with open("data/opa_properties_public.geojson", "w") as f:
#     f.write(resp.text)

import requests

url    = "https://phl.carto.com/api/v2/sql"
params = {
    "q": """
      SELECT
        *,
        ST_AsGeoJSON(the_geom)::json AS geometry
      FROM opa_properties_public
      WHERE assessment_date >= '2022-01-01'
    """,
    "format": "GeoJSON"
}

resp = requests.get(url, params=params)
resp.raise_for_status()

props = resp.json()
print(f"Retrieved {len(resp['features'])} features.")
with open("data/opa_properties_public.geojson", "w") as f:
    f.write(props.text)
    

In [ ]:
import requests

# Endpoint and parameters
url = "https://phl.carto.com/api/v2/sql"
params = {
    "q": "SELECT *, ST_AsGeoJSON(the_geom)::json AS the_geom_geojson FROM public_cases_fc WHERE requested_datetime >= '2022-01-01'",
    "format": "GeoJSON"
}

# Send the GET request
resp = requests.get(url, params=params)
resp.raise_for_status()  # will raise if response code is 4xx/5xx

# Parse as GeoJSON
calls = resp.json()
print(f"Retrieved {len(calls['features'])} features.")
with open("data/311.geojson", "w") as f:
    f.write(resp.text)


Retrieved 1831001 features.


In [8]:
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import folium
import numpy as np

In [9]:
# 1. Load property polygons

props = gpd.read_file("data/opa_properties_public.geojson").to_crs(3857)

# 2. Load 311 point data and spatial‑join to parcels

calls = gpd.read_file("data/311.geojson").to_crs(3857)

joined = gpd.sjoin(props, calls[['geometry','service_name']], how="left")

In [10]:
props

,cartodb_id,assessment_date,basements,beginning_point,book_and_page,building_code,building_code_description,category_code,category_code_description,census_tract,...,view_type,year_built,year_built_estimate,zip_code,zoning,pin,building_code_new,building_code_description_new,objectid,geometry
0,1,2024-06-06 00:00:00+00:00,H,"65'5"" W OF 18TH ST",54421066,R50,ROW B/GAR 3 STY MASONRY,1,SINGLE FAMILY,373,...,I,1960,Y,19145,RSA5,1001474749,21,ROW OLD STYLE,625493651,POINT (-8368801.614 4853114.259)
1,2,2024-06-06 00:00:00+00:00,A,117' W OF 18TH ST,54421184,O50,ROW 3 STY MASONRY,1,SINGLE FAMILY,140,...,I,1915,Y,19121,RSA5,1001520353,22,ROW TYPICAL,625493652,POINT (-8367444.407 4862158.067)
2,3,2024-06-06 00:00:00+00:00,D,"135'11 1/2""N MIFFLIN",54421270,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,I,1950,Y,19145,RM1,1001646206,23,ROW POST WAR,625493653,POINT (-8370301.088 4855801.554)
3,4,2024-06-06 00:00:00+00:00,D,"26'5 1/2""N MIFFLIN ST",54421269,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,I,1950,Y,19145,RM1,1001646217,23,ROW POST WAR,625493654,POINT (-8370308.435 4855758.874)
4,5,2024-06-06 00:00:00+00:00,None,"297'8"" W 17TH ST",54421205,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,140,...,None,1915,None,19130,RM4,1001242337,806,APARTMENTS - BLT AS RESID,625493655,POINT (-8367386.855 4861872.922)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
582773,1416,2024-06-06 00:00:00+00:00,None,407' W 50TH ST,54413155,H30,SEMI/DET 2 STY MASONRY,1,SINGLE FAMILY,85,...,I,1925,Y,19139,RSA5,1001140443,32,TWIN CONVENTIONAL,625494965,POINT (-8373760.007 4859901.535)
582774,1417,2024-06-06 00:00:00+00:00,None,210' W 50TH ST,54413156,O30,ROW 2 STY MASONRY,1,SINGLE FAMILY,73,...,I,1925,None,19143,RM1,1001217562,24,ROW PORCH FRONT,625494966,POINT (-8373689.542 4858083.727)
582775,1418,2024-06-06 00:00:00+00:00,F,"136'6"" N. ANNSBURY ST",54413336,R30,ROW B/GAR 2 STY MASONRY,1,SINGLE FAMILY,289,...,I,1930,Y,19120,RM1,1001232070,24,ROW PORCH FRONT,625494967,POINT (-8361284.654 4868502.519)
582776,1419,2024-06-06 00:00:00+00:00,F,278' W 46 ST,54413480,K50,S/D W/B GAR 3 STY MASONRY,1,SINGLE FAMILY,79,...,I,1925,None,19143,RSA3,1001313055,36,TWIN OLD STYLE,625494968,POINT (-8372982.218 4858811.943)


In [11]:
calls

,cartodb_id,objectid,service_request_id,subject,status,status_notes,service_name,service_code,agency_responsible,service_notice,requested_datetime,updated_datetime,expected_datetime,closed_datetime,address,zipcode,media_url,lat,lon,geometry
0,513308,3477876,14735655,Maintenance Complaint,Closed,None,Maintenance Complaint,None,Community Life Improvement Program,90 Business Days,2022-02-14 19:45:38+00:00,2023-02-16 08:09:46+00:00,2022-07-05 04:00:00+00:00,2022-03-24 13:11:52+00:00,2925 MASTER ST,19121,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.977715,-75.183728,POINT (-8369414.317 4862704.412)
1,536719,3416151,14653360,Will the City pickup my trash on a Holiday?,Closed,Question Answered,Information Request,SR-IR01,Streets Department,None,2022-01-05 05:27:33+00:00,2022-01-05 05:27:33+00:00,NaT,2022-01-05 05:27:33+00:00,None,None,None,NaN,NaN,None
2,537325,3416152,14653361,Will the City pickup my trash on a Holiday?,Closed,Question Answered,Information Request,SR-IR01,Streets Department,None,2022-01-05 05:28:08+00:00,2022-01-05 05:28:09+00:00,NaT,2022-01-05 05:28:08+00:00,None,None,None,NaN,NaN,None
3,537388,3420018,14658595,How do I contact the Department of Revenue?,Closed,Question Answered,Information Request,SR-IR01,Revenue,None,2022-01-07 03:02:25+00:00,2022-01-07 03:02:51+00:00,NaT,2022-01-07 03:02:25+00:00,None,None,None,NaN,NaN,None
4,538966,3420146,14658765,What is the PHL City ID?,Closed,Question Answered,Information Request,SR-IR01,Managing Director's Office- MDO,None,2022-01-07 03:44:04+00:00,2022-01-07 03:44:44+00:00,NaT,2022-01-07 03:44:04+00:00,None,None,None,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1830996,2392291,5242749,17602694,Street Light Out,Open,None,Miscellaneous,SR-MI01,Philly311 Contact Center,None,2025-05-16 11:11:52+00:00,2025-05-16 11:17:01+00:00,NaT,NaT,Tacony Creek Park,19124,None,40.026808,-75.113360,POINT (-8361580.987 4869838.71)
1830997,2392292,5242750,17602695,Abandoned Vehicle | Test record Please ignore,Closed,None,Abandoned Vehicle,SR-PD01,Police Department,120 Business Days,2025-05-16 11:16:22+00:00,2025-05-16 11:21:26+00:00,2025-11-20 05:00:00+00:00,2025-05-16 11:17:58+00:00,1600 N AMERICAN ST,19130,None,39.961714,-75.173345,POINT (-8368258.487 4860380.221)
1830998,2392293,5242751,17602696,Graffiti Removal | Test record please ignore,Closed,Other,Graffiti Removal,SR-CL01,Community Life Improvement Program,7 Business Days,2025-05-16 11:21:44+00:00,2025-05-16 11:26:51+00:00,2025-05-27 04:00:00+00:00,2025-05-16 11:26:39+00:00,CALLOWHILL ST & N CHRISTOPHER COLUMBUS BLVD,19123,None,39.956832,-75.138710,POINT (-8364402.936 4859671.205)
1830999,2392294,5242752,17602697,Graffiti Removal | Test record please ignore,Closed,None,Graffiti Removal,SR-CL01,Community Life Improvement Program,7 Business Days,2025-05-16 11:25:23+00:00,2025-05-16 11:30:29+00:00,2025-05-27 04:00:00+00:00,2025-05-16 11:26:43+00:00,CHESTNUT ST & S 23RD ST,19103,None,39.952592,-75.178176,POINT (-8368796.271 4859055.468)


In [12]:
joined

,cartodb_id,assessment_date,basements,beginning_point,book_and_page,building_code,building_code_description,category_code,category_code_description,census_tract,...,year_built_estimate,zip_code,zoning,pin,building_code_new,building_code_description_new,objectid,geometry,index_right,service_name
0,1,2024-06-06 00:00:00+00:00,H,"65'5"" W OF 18TH ST",54421066,R50,ROW B/GAR 3 STY MASONRY,1,SINGLE FAMILY,373,...,Y,19145,RSA5,1001474749,21,ROW OLD STYLE,625493651,POINT (-8368801.614 4853114.259),NaN,NaN
1,2,2024-06-06 00:00:00+00:00,A,117' W OF 18TH ST,54421184,O50,ROW 3 STY MASONRY,1,SINGLE FAMILY,140,...,Y,19121,RSA5,1001520353,22,ROW TYPICAL,625493652,POINT (-8367444.407 4862158.067),NaN,NaN
2,3,2024-06-06 00:00:00+00:00,D,"135'11 1/2""N MIFFLIN",54421270,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,Y,19145,RM1,1001646206,23,ROW POST WAR,625493653,POINT (-8370301.088 4855801.554),NaN,NaN
3,4,2024-06-06 00:00:00+00:00,D,"26'5 1/2""N MIFFLIN ST",54421269,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,Y,19145,RM1,1001646217,23,ROW POST WAR,625493654,POINT (-8370308.435 4855758.874),NaN,NaN
4,5,2024-06-06 00:00:00+00:00,None,"297'8"" W 17TH ST",54421205,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,140,...,None,19130,RM4,1001242337,806,APARTMENTS - BLT AS RESID,625493655,POINT (-8367386.855 4861872.922),752984.0,Information Request
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
582773,1416,2024-06-06 00:00:00+00:00,None,407' W 50TH ST,54413155,H30,SEMI/DET 2 STY MASONRY,1,SINGLE FAMILY,85,...,Y,19139,RSA5,1001140443,32,TWIN CONVENTIONAL,625494965,POINT (-8373760.007 4859901.535),NaN,NaN
582774,1417,2024-06-06 00:00:00+00:00,None,210' W 50TH ST,54413156,O30,ROW 2 STY MASONRY,1,SINGLE FAMILY,73,...,None,19143,RM1,1001217562,24,ROW PORCH FRONT,625494966,POINT (-8373689.542 4858083.727),NaN,NaN
582775,1418,2024-06-06 00:00:00+00:00,F,"136'6"" N. ANNSBURY ST",54413336,R30,ROW B/GAR 2 STY MASONRY,1,SINGLE FAMILY,289,...,Y,19120,RM1,1001232070,24,ROW PORCH FRONT,625494967,POINT (-8361284.654 4868502.519),NaN,NaN
582776,1419,2024-06-06 00:00:00+00:00,F,278' W 46 ST,54413480,K50,S/D W/B GAR 3 STY MASONRY,1,SINGLE FAMILY,79,...,None,19143,RSA3,1001313055,36,TWIN OLD STYLE,625494968,POINT (-8372982.218 4858811.943),NaN,NaN


In [13]:
# 3. Feature engineering: count complaints per parcel
feat = (joined.groupby("parcel_number").service_name.count().rename("complaint_ct").reset_index())

df = props.merge(feat, on="parcel_number", how="left")

df["complaint_ct"] = df["complaint_ct"].fillna(0).astype(int)

df

,cartodb_id,assessment_date,basements,beginning_point,book_and_page,building_code,building_code_description,category_code,category_code_description,census_tract,...,year_built,year_built_estimate,zip_code,zoning,pin,building_code_new,building_code_description_new,objectid,geometry,complaint_ct
0,1,2024-06-06 00:00:00+00:00,H,"65'5"" W OF 18TH ST",54421066,R50,ROW B/GAR 3 STY MASONRY,1,SINGLE FAMILY,373,...,1960,Y,19145,RSA5,1001474749,21,ROW OLD STYLE,625493651,POINT (-8368801.614 4853114.259),0
1,2,2024-06-06 00:00:00+00:00,A,117' W OF 18TH ST,54421184,O50,ROW 3 STY MASONRY,1,SINGLE FAMILY,140,...,1915,Y,19121,RSA5,1001520353,22,ROW TYPICAL,625493652,POINT (-8367444.407 4862158.067),0
2,3,2024-06-06 00:00:00+00:00,D,"135'11 1/2""N MIFFLIN",54421270,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,1950,Y,19145,RM1,1001646206,23,ROW POST WAR,625493653,POINT (-8370301.088 4855801.554),0
3,4,2024-06-06 00:00:00+00:00,D,"26'5 1/2""N MIFFLIN ST",54421269,O10,ROW 1 STY MASONRY,1,SINGLE FAMILY,36,...,1950,Y,19145,RM1,1001646217,23,ROW POST WAR,625493654,POINT (-8370308.435 4855758.874),0
4,5,2024-06-06 00:00:00+00:00,None,"297'8"" W 17TH ST",54421205,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,140,...,1915,None,19130,RM4,1001242337,806,APARTMENTS - BLT AS RESID,625493655,POINT (-8367386.855 4861872.922),1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
582773,1416,2024-06-06 00:00:00+00:00,None,407' W 50TH ST,54413155,H30,SEMI/DET 2 STY MASONRY,1,SINGLE FAMILY,85,...,1925,Y,19139,RSA5,1001140443,32,TWIN CONVENTIONAL,625494965,POINT (-8373760.007 4859901.535),0
582774,1417,2024-06-06 00:00:00+00:00,None,210' W 50TH ST,54413156,O30,ROW 2 STY MASONRY,1,SINGLE FAMILY,73,...,1925,None,19143,RM1,1001217562,24,ROW PORCH FRONT,625494966,POINT (-8373689.542 4858083.727),0
582775,1418,2024-06-06 00:00:00+00:00,F,"136'6"" N. ANNSBURY ST",54413336,R30,ROW B/GAR 2 STY MASONRY,1,SINGLE FAMILY,289,...,1930,Y,19120,RM1,1001232070,24,ROW PORCH FRONT,625494967,POINT (-8361284.654 4868502.519),0
582776,1419,2024-06-06 00:00:00+00:00,F,278' W 46 ST,54413480,K50,S/D W/B GAR 3 STY MASONRY,1,SINGLE FAMILY,79,...,1925,None,19143,RSA3,1001313055,36,TWIN OLD STYLE,625494968,POINT (-8372982.218 4858811.943),0


In [14]:
df[df['complaint_ct'] > 0]

,cartodb_id,assessment_date,basements,beginning_point,book_and_page,building_code,building_code_description,category_code,category_code_description,census_tract,...,year_built,year_built_estimate,zip_code,zoning,pin,building_code_new,building_code_description_new,objectid,geometry,complaint_ct
4,5,2024-06-06 00:00:00+00:00,None,"297'8"" W 17TH ST",54421205,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,140,...,1915,None,19130,RM4,1001242337,806,APARTMENTS - BLT AS RESID,625493655,POINT (-8367386.855 4861872.922),1
6,7,2024-06-06 00:00:00+00:00,None,SWC 17TH ST,54421209,S30,ROW W/OFF STR 2 STY MASON,3,MIXED USE,30,...,1925,None,19146,CMX1,1001179408,821,ROW MIXED-COM/RES-BLT AS COM,625493657,POINT (-8368229.655 4856093.348),8
11,12,2024-06-06 00:00:00+00:00,None,SWC JUNIPER ST,54421040,590,RES CONDO 5+ STY MASONRY,1,SINGLE FAMILY,9,...,1900,Y,19107,CMX5,1001332191,08,CONDO FLATS,625493662,POINT (-8367182.361 4858359.179),4
27,28,2024-06-06 00:00:00+00:00,C,SWC OF FERNON ST,54420739,U50,ROW CONV/APT 3 STY MASON,2,MULTI FAMILY,28,...,2011,None,19148,RSA5,1001600416,25,ROW MODERN,625493678,POINT (-8366281.675 4855527.043),1
50,51,2024-06-06 00:00:00+00:00,None,878.694' NW STONEY LN,54421271,531,RES CONDO 2 STY MAS+OTH,1,SINGLE FAMILY,345,...,1980,Y,19115,RTA1,1001387466,27,TWIN POST WAR,625493631,POINT (-8353751.553 4878410.489),1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
582641,582560,2024-06-06 00:00:00+00:00,None,135' N DREXEL RD,0000000,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,117,...,1927,None,19131,RSA3,1001664663,828,APTS - GARDEN,626075776,POINT (-8375759.751 4864656.305),1
582655,582541,2024-06-06 00:00:00+00:00,D,60' NE WILMOT ST,0000000,H30,SEMI/DET 2 STY MASONRY,1,SINGLE FAMILY,183,...,1920,None,19137,RSA5,1001062569,32,TWIN CONVENTIONAL,626076264,POINT (-8357220.936 4866114.773),6
582682,582586,2024-06-06 00:00:00+00:00,None,120' S PINE ST,0000000,CA0,APTS 5-50 UNITS MASONRY,14,APARTMENTS > 4 UNITS,11,...,1900,None,19147,RM1,1001608402,804,APARTMENTS - LOW RISE,626076278,POINT (-8366458.228 4857828.319),1
582703,582625,2024-06-06 00:00:00+00:00,1,"113'2"" NW EMERY S\",None,P76,ROW W/GAR 4 STY FRAME,1,SINGLE FAMILY,143,...,2021,None,19125,RSA5,1001680894,25,ROW MODERN,626076284,POINT (-8363382.582 4861407.092),1


In [15]:
df["shape_area"] = props.geometry.area

In [16]:
df_cleaned = df.drop(index=df[df["sale_price"].isnull()].index)

In [ ]:
import xgboost as xgb

# 4. Simple model: predict log(sale_price) from complaint_ct & area
X = df_cleaned[["complaint_ct", "shape_area"]]

y = np.log1p(df_cleaned["sale_price"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

# m = RandomForestRegressor(n_estimators=200).fit(X_train, y_train)

# print("Test R²:", m.score(X_test, y_test))

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.7,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train,
          eval_set=[(X_train, y_train),(X_test, y_test)],
          early_stopping_rounds=50,
          verbose=False)

print("XGB R²:", model.score(X_test, y_test))

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/ColeHoffman/dev/gis_phl/gis-phl/venv/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <948FC7F9-7446-3923-BB9F-85890E78C765> /Users/ColeHoffman/dev/gis_phl/gis-phl/venv/lib/python3.11/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew

In [ ]:
# 5. Visualise predictions on an interactive map

df_cleaned["pred_price"] = np.expm1(m.predict(X))

# reproject for mapping
gd = df_cleaned.to_crs(4326)

# pull out only the fields Folium needs:
geojson = {
    "type": "FeatureCollection",
    "features": []
}
for _, row in gd.iterrows():
    geom = row.geometry
    if geom is None:
        continue   # skip parcels with no geometry
    geojson["features"].append({
        "type": "Feature",
        "geometry": geom.__geo_interface__,
        "properties": {
            "parcel_number": row.parcel_number,
            "pred_price": row.pred_price
        }
    })

In [63]:
import json
with open("data/philly_parcels.geojson", "w") as f:
    json.dump(geojson, f)

In [64]:
df_cleaned.to_csv("data/philly_parcels.csv", index=False)

In [ ]:
# Basic but does not scale well 
mmap = folium.Map(location=[39.95,-75.17], zoom_start=11, tiles="cartodbpositron")

folium.Choropleth(geo_data=geojson,
                  data=df_cleaned,
                  columns=["parcel_number","pred_price"],
                  key_on="feature.properties.parcel_number",
                  fill_color="YlGnBu",
                  legend_name="Predicted Sale Price").add_to(mmap)

mmap.save("philly_parcels.html")

# Scaled Solution

In [ ]:
%%bash
tippecanoe --force -o parcels.mbtiles -Z 10 -z 16 --drop-densest-as-needed data/philly_parcels.geojson

# install a lightweight MBTiles tile server
npm install -g tileserver-gl-light

# point it at your tileset
tileserver-gl-light parcels.mbtiles

In [65]:
# from IPython.display import IFrame
# IFrame("philly_parcels.html", width=800, height=600)

In [ ]:
%%bash
python -m http.server 8000